In [ ]:
import numpy as np
import albumentations as A
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from auxiliary.numpy_dataset import NumpyDataset

## Dataset Load

In [ ]:
DATA_DIR    = "Pokemon"
IMG_SIZE    = (64, 64)
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
N_FOLDS     = 5
RANDOM_SEED = 42
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
 
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Using device: {DEVICE}")

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
train_transform = A.Compose([
    A.Resize(64, 64),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

## Data Preprocessing

In [ ]:
X_train = np.load('Pokemon/X_train.npy')   
y_train = np.load('Pokemon/y_train.npy')

X_val = np.load("Pokemon/X_val.npy")
y_val = np.load("Pokemon/y_val.npy")

X_test = np.load("Pokemon/X_test.npy")
y_test = np.load("Pokemon/y_test.npy")

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

le = LabelEncoder()
y_train_idx = le.fit_transform(y_train)  
y_val_idx   = le.transform(y_val)        
y_test_idx  = le.transform(y_test)


y = le.fit_transform(y)
train_loader = DataLoader(NumpyDataset(X_train, y_train_idx, transform=train_transform), 
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NumpyDataset(X_val, y_val_idx, transform=val_transform), 
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(NumpyDataset(X_test, y_test_idx, transform=val_transform), 
                          batch_size=BATCH_SIZE, shuffle=False)

#print(y[:5])

## Build Neural Network

In [ ]:
#num_classes = len(np.unique(y))
    #layers.Dense(128, activation='relu'),
    #layers.Dense(num_classes, activation='softmax')

#Is still needed??
 #layers.Dense(512, activation='relu', input_shape=(X.shape[1],)),
    #layers.Dense(256, activation='relu'),

model = keras.Sequential([
    # Convolution Layer 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    layers.MaxPooling2D((2,2)),
    
    # Convolution Layer 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Convolution Layer 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Flatten
    layers.Flatten(),
    
    # Dense Layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    
    # Output Layer (number of classes)
    layers.Dense(len(np.unique(y_train)), activation='softmax')
])

#Compile
#Still need?
#(learning_rate=0.001)
model.compile(
    optimizer=keras.optimizers.Adam,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## Train

In [ ]:
#Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32
)

## Plot Results

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()
all_preds = []
all_labels = []
all_probs = [] # Needed for ROC-AUC

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        
        # Get probabilities using Softmax
        probs = F.softmax(outputs, dim=1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())


cm = confusion_matrix(y_test, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)

In [ ]:
# Accuracy
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.title('Accuracy')
plt.show()

# Loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title('Loss')
plt.show()

## Evaluate

In [ ]:
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.4f}")